# Render PiramidaStickman (Edukasi Stickman) di Google Colab (T4 GPU)

Video edukasi stickman: **Logistik Pembangunan Piramida Giza (Catatan Mandor Merer)**.
Spesifikasi: 1920x1080 @ 30fps, durasi 500 detik (15.000 frame), 151 scene visual.

### Langkah Cepat Render:
1. Pastikan runtime Colab memakai **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar hasil video otomatis tersimpan di Google Drive Anda.
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi dependensi, Chromium, Node.js 20, dan clone repo.
4. *(Opsional)* Jalankan **Sel 3 (Uji Coba 10 Detik)** untuk verifikasi kilat.
5. Jalankan **Sel 4 (Render Video Full)** untuk merender full video 500 detik.
6. Jalankan **Sel 5 (Download Video)** untuk mengunduh file `.mp4` langsung ke komputer/laptop Anda.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/remotion-out


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap render PiramidaStickman."


In [ ]:
# Sel 3: (Opsional) Uji Coba Cepat 10 Detik Pertama (300 frames) - 1 baris progress anti-lag
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)
colab_render.run_render(
    'PiramidaStickman',
    f'{out_dir}/test-piramida-10s.mp4',
    extra_args=['--frames=0-299'],
    cwd='/content/vidio',
)


In [ ]:
# Sel 4: Render Video Full: PiramidaStickman (1920x1080 @ 30fps, 15.000 frames, ~8.33 menit)
import importlib, os, subprocess, sys

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
sys.path.insert(0, '/content/vidio/scripts')
import colab_render
importlib.reload(colab_render)
colab_render.run_render('PiramidaStickman', f'{out_dir}/piramida-stickman-1080p.mp4', cwd='/content/vidio')


In [ ]:
# Sel 5: Download langsung piramida-stickman-1080p.mp4 ke laptop
from google.colab import files
import os

for p in [
    '/content/drive/MyDrive/remotion-out/piramida-stickman-1080p.mp4',
    '/content/remotion-out/piramida-stickman-1080p.mp4',
]:
    if os.path.exists(p):
        print('Mengunduh:', p)
        files.download(p)
        break
else:
    print('File piramida-stickman-1080p.mp4 belum ditemukan. Jalankan sel render PiramidaStickman dulu.')
